In [1]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3
import numpy as np

import re
from io import BytesIO
from urllib.parse import urlparse
from typing import List, Optional, Tuple

from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from functools import partial

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [4]:
print(SSP_DIR_PATH)

c:\Users\pkane\sspla\ssp_louisiana\metamodel\data\ssp


In [5]:
run_id = "sisepuede_run_2025-10-07t13;30;14.193421"

In [6]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, run_id)

In [7]:
SIM_PARQUET_DIR_PATH = os.path.join(SIMULATION_DIR_PATH, "parquets")

In [8]:
decomposed_df = pd.read_parquet(os.path.join(SIM_PARQUET_DIR_PATH, "decomposed_combined.parquet"))

ArrowMemoryError: malloc of size 8388608 failed

In [ ]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'natural_gas_liquid',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [ ]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [ ]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': decomposed_df['primary_id'],
    'time_period': decomposed_df['time_period']
}, index=decomposed_df.index)

# Helper alias for tricky fuel names
alias = {
    'natural_gas_liquid': ['natural_gas_liquid', 'natural_gas_liquids'],
    'biomass': ['biomass', 'solid_biomass'],   # <-- added
}

for fuel in relevant_fuels:
    fuel_keys = alias.get(fuel, [fuel])  # use aliases everywhere for this fuel

    # ------------------------------------------------------------
    # ENTc (fuel-only) — handled OUTSIDE sector loop
    # ------------------------------------------------------------
    entc_found_col = None
    for k in fuel_keys:
        cand = f'energy_demand_enfu_subsector_total_pj_entc_fuel_{k}'
        if cand in decomposed_df.columns:
            entc_found_col = cand
            break
    if entc_found_col is not None:
        ind_fuel_demand_by_sector[entc_found_col] = decomposed_df[entc_found_col]

    # ------------------------------------------------------------
    # Efficiency column (accept any alias)
    # ------------------------------------------------------------
    eff_cols = [
        c for c in decomposed_df.columns
        if any(c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{k}') for k in fuel_keys)
    ]
    if not eff_cols:
        continue
    fuel_efficiency = decomposed_df[eff_cols[0]]

    # ------------------------------------------------------------
    # Sector loop (unchanged logic; now alias-aware for fractions)
    # ------------------------------------------------------------
    for sector in sectors:
        sector_dem_cols = [c for c in decomposed_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [
            c for c in decomposed_df.columns
            if any(f'frac_inen_energy_{sector}_{k}' in c for k in fuel_keys)
        ]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = decomposed_df[sector_dem_cols[0]]
            sector_fuel_fraction = decomposed_df[sector_fuel_fraction_cols[0]]

            has_any_demand = ((sector_fuel_fraction * sector_total_demand).fillna(0) != 0).any()
            if has_any_demand or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed + deltas
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(decomposed_df['primary_id']).transform('first')
                )
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline - sector_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )

In [ ]:
del(decomposed_df)

In [ ]:
ind_fuel_demand_by_sector_100k = ind_fuel_demand_by_sector
del(ind_feul_demand_by_sector)

In [ ]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [ ]:
run_id="sisepuede_run_2025-09-18t09;19;22.726476"

In [ ]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, run_id)

In [ ]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))

In [ ]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "sisepuede_results_IDE_2025-09-18t09;19;22.726476.csv"))

In [ ]:
attr_primary_df = attr_primary_df[attr_primary_df["strategy_id"].isin([6004])]

In [ ]:
wide_inputs_outputs_df = wide_inputs_outputs_df[wide_inputs_outputs_df["primary_id"].isin(attr_primary_df["primary_id"].unique())]

In [ ]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'natural_gas_liquid',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [ ]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [ ]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': wide_inputs_outputs_df['primary_id'],
    'time_period': wide_inputs_outputs_df['time_period']
}, index=wide_inputs_outputs_df.index)

# Helper alias for tricky fuel names
alias = {
    'natural_gas_liquid': ['natural_gas_liquid', 'natural_gas_liquids'],
    'biomass': ['biomass', 'solid_biomass'],   # <-- added
}

for fuel in relevant_fuels:
    fuel_keys = alias.get(fuel, [fuel])  # use aliases everywhere for this fuel

    # ------------------------------------------------------------
    # ENTc (fuel-only) — handled OUTSIDE sector loop
    # ------------------------------------------------------------
    entc_found_col = None
    for k in fuel_keys:
        cand = f'energy_demand_enfu_subsector_total_pj_entc_fuel_{k}'
        if cand in wide_inputs_outputs_df.columns:
            entc_found_col = cand
            break
    if entc_found_col is not None:
        ind_fuel_demand_by_sector[entc_found_col] = wide_inputs_outputs_df[entc_found_col]

    # ------------------------------------------------------------
    # Efficiency column (accept any alias)
    # ------------------------------------------------------------
    eff_cols = [
        c for c in wide_inputs_outputs_df.columns
        if any(c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{k}') for k in fuel_keys)
    ]
    if not eff_cols:
        continue
    fuel_efficiency = wide_inputs_outputs_df[eff_cols[0]]

    # ------------------------------------------------------------
    # Sector loop (unchanged logic; now alias-aware for fractions)
    # ------------------------------------------------------------
    for sector in sectors:
        sector_dem_cols = [c for c in wide_inputs_outputs_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [
            c for c in wide_inputs_outputs_df.columns
            if any(f'frac_inen_energy_{sector}_{k}' in c for k in fuel_keys)
        ]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = wide_inputs_outputs_df[sector_dem_cols[0]]
            sector_fuel_fraction = wide_inputs_outputs_df[sector_fuel_fraction_cols[0]]

            has_any_demand = ((sector_fuel_fraction * sector_total_demand).fillna(0) != 0).any()
            if has_any_demand or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed + deltas
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(wide_inputs_outputs_df['primary_id']).transform('first')
                )
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline - sector_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )

In [ ]:
ind_fuel_demand_by_sector_1k = ind_fuel_demand_by_sector

In [ ]:
print('100k dimensions: ' ind_fuel_demand_by_sector_100k.shape)
print('1k dimensions: ' ind_fuel_demand_by_sector_1k.shape)

In [9]:
columns_100k = list(ind_fuel_demand_by_sector_100k.columns)
columns_1k = list(ind_fuel_demand_by_sector_1k.columns)

missing_from_100k = [value for value in columns_100k if value not in columns_1k]
missing_from_1k = [value for value in columns_1k if value not in columns_100k]

NameError: name 'ind_fuel_demand_by_sector_100k' is not defined

In [ ]:
print(missing_from_100k)

In [ ]:
print(missing_from_1k)

In [ ]:
columns_in_both = [value for value in columns_100k if value in columns_1k]


In [ ]:
for col in columns_in_both:
    print(col)
    print('100k average: ',  ind_fuel_demand_by_sector_100k[col].mean())
    print('1k average: ',  ind_fuel_demand_by_sector_1k[col].mean())

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
x=ind_fuel_demand_by_sector_100k[[columns_in_both]].mean()
y=ind_fuel_demand_by_sector_100k[[columns_in_both]].mean()
plt.scatter(x, y)